In [30]:
import pandas as pd

df = pd.read_csv("../../../../Merge/final_selected_data.csv")
df.head()

,room_type_id,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,...,wifi_miễn_phí,không_hoàn_tiền,miễn_phí_hủy,vào_hồ_bơi_miễn_phí,đã_kèm_bữa_sáng,hotel_id,room_room_type_name,hotel_name,hotel_address,region
0,1,1,0,0,0,0,0,0,0,0,...,1,0,1,0,0,71897952,Phòng Deluxe Có Giường Cỡ King (Deluxe King Room),Nha Nghi Nhung - Nhung Motel,"73 Đoàn Thị Điểm, Bà Rịa, Bà Rịa, Việt Nam",Bà Rịa
1,2,0,1,0,0,0,0,0,0,0,...,1,0,1,0,0,49685029,Phòng Tiêu Chuẩn (Standard Room),Nhà nghỉ Ruby Bà Rịa (Ruby Motel Bà Rịa),"KDC Baria City Gate, Long Huong Ward, Ba Ria C...",Bà Rịa
2,3,0,1,0,0,0,0,0,0,0,...,1,0,1,0,0,49685029,Phòng gia đình có ban công (Family Room with B...,Nhà nghỉ Ruby Bà Rịa (Ruby Motel Bà Rịa),"KDC Baria City Gate, Long Huong Ward, Ba Ria C...",Bà Rịa
3,4,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,65481766,Phòng Có Giường Cỡ King Với Ban Công (King Roo...,Baly Hotel Bà Rịa City (Baly Hotel Ba Ria City),"QL51, Bà Rịa, Bà Rịa, Việt Nam",Bà Rịa
4,5,1,0,1,0,0,0,0,0,0,...,1,0,0,0,0,5808626,Phòng Studio Executive (Studio Executive),Citadines Central Bình Dương (Citadines Centra...,"Số 328C, Đại lộ B nh Dương, Khu phố Hưng Lộc, ...",Bình Dương


In [31]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso


from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

In [32]:
df.drop(columns=["room_type_id", "hotel_id", "room_room_type_name", "hotel_name", "hotel_address"], inplace=True, errors='ignore')
df = df.dropna()

In [ ]:
TARGET_COL = "price_option_price"

numerical_cols = [
    'flexibility_score', 'sqm', 'bathroom_count', 
    'bedroom_count', 'adults_number', 'children_number'
]

binary_cols = [
    'large_double_bed', 'large_bed', 'single_bed', 'sofa_bed', 'double_bed',
    'small_double_bed', 'king_size_bed', 'futon_mattress', 'bunk_bed',
    'is_private_bathroom', 'balcony-terrace', 'closet', 'air_conditioning', 
    'hair_dryer', 'complimentary-bottled-water', 'bathtub', 'shower', 
    'refrigerator', 'high-floor', 'dressing-room', 'ground-floor', 'private-pool',
    'top-floor', 'complimentary-instant-coffee', 'electric-blanket', 
    'free-welcome-drink', 'low-floor', 'complimentary-tea', 'coffee-tea-maker', 
    'hot-spring-access', 'bãi_đậu_xe', 'phòng_tập', 'wifi_miễn_phí', 'không_hoàn_tiền', 
    'miễn_phí_hủy', 'vào_hồ_bơi_miễn_phí', 'đã_kèm_bữa_sáng'
]

categorical_cols = [
    "views",   # ví dụ: sea, mountain, city
    "region"
]

In [34]:
X = df[numerical_cols + binary_cols + categorical_cols]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [35]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"  # giữ numerical + binary
)

def train_and_evaluate(model, model_name):
    pipeline = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    return {
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

In [36]:
lin_reg = LinearRegression()

result_linear = train_and_evaluate(
    lin_reg,
    "Linear Regression"
)

In [37]:
ridge = Ridge(alpha=1.0)

result_ridge = train_and_evaluate(
    ridge,
    "Ridge Regression"
)

C:\Users\Admin\AppData\Roaming\Python\Python313\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.77419e-18): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


In [38]:
lasso = Lasso(alpha=0.001, max_iter=10000)

result_lasso = train_and_evaluate(
    lasso,
    "Lasso Regression"
)

In [41]:
results = pd.DataFrame([
    result_linear,
    result_ridge,
    result_lasso
])

results

,model,MAE,RMSE,R2
0,Linear Regression,2.719100e-09,4.082947e-09,1.0
1,Ridge Regression,6.046983e-09,8.789492e-09,1.0
2,Lasso Regression,4.096529e-10,9.105068e-10,1.0


In [47]:
results_fmt = results.copy()

results_fmt["MAE"] = results_fmt["MAE"].apply(lambda x: f"{x:.3e}")
results_fmt["RMSE"] = results_fmt["RMSE"].apply(lambda x: f"{x:.3e}")
results_fmt["R2"] = results_fmt["R2"].apply(lambda x: f"{x:.4f}")

display(results_fmt)

,model,MAE,RMSE,R2
0,Linear Regression,2.719e-09,4.083e-09,1.0000
1,Ridge Regression,6.047e-09,8.789e-09,1.0000
2,Lasso Regression,4.097e-10,9.105e-10,1.0000
